In [40]:
# install dependences
!pip install zenodo_get torch_geometric h5py -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [41]:
import torch
torch_version = torch.__version__.split('+')[0]
print(f"PyTorch version: {torch_version}")

!pip install pyg-lib -f https://data.pyg.org/whl/torch-{torch_version}+cpu.html

PyTorch version: 2.11.0
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.3 MB/s eta 0:00:00


In [12]:
# mount Google Drive to ensure data persistence.
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/jet_tagging'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# download dataset
!zenodo_get 2603256 -o /content/data/

INFO: Output directory: /content/data
INFO: Title: Top Quark Tagging Reference Dataset
INFO: Total size: 1.7 GB
INFO: Number of files: 3
INFO: val.h5 is already downloaded correctly.
INFO: test.h5 is already downloaded correctly.
INFO: train.h5 is already downloaded correctly.
SUCCESS: All specified files have been processed.


In [14]:
import h5py

def explore_hdf5(f, indent=0):
    for key in f.keys():
        item = f[key]
        prefix = "  " * indent
        if isinstance(item, h5py.Group):
            print(f"{prefix}Group: {key}/")
            explore_hdf5(item, indent + 1)
        elif isinstance(item, h5py.Dataset):
            print(f"{prefix}Dataset: {key} | shape={item.shape} | dtype={item.dtype}")

with h5py.File('/content/data/train.h5', 'r') as f:
    explore_hdf5(f)

Group: table/
  Group: _i_table/
    Group: index/
      Dataset: abounds | shape=(1152,) | dtype=int64
      Dataset: bounds | shape=(9, 127) | dtype=int64
      Dataset: indices | shape=(9, 131072) | dtype=uint32
      Dataset: indicesLR | shape=(131072,) | dtype=uint32
      Dataset: mbounds | shape=(1152,) | dtype=int64
      Dataset: mranges | shape=(9,) | dtype=int64
      Dataset: ranges | shape=(9, 2) | dtype=int64
      Dataset: sorted | shape=(9, 131072) | dtype=int64
      Dataset: sortedLR | shape=(131201,) | dtype=int64
      Dataset: zbounds | shape=(1152,) | dtype=int64
  Dataset: table | shape=(1211000,) | dtype=[('index', '<i8'), ('values_block_0', '<f4', (804,)), ('values_block_1', '<i8', (2,))]


In [15]:
import pandas as pd

# Load only 5 rows to inspect
df = pd.read_hdf('/content/data/train.h5', key='table', stop=5)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:\n", df.dtypes.value_counts())

Shape: (5, 806)
Columns: ['E_0', 'PX_0', 'PY_0', 'PZ_0', 'E_1', 'PX_1', 'PY_1', 'PZ_1', 'E_2', 'PX_2', 'PY_2', 'PZ_2', 'E_3', 'PX_3', 'PY_3', 'PZ_3', 'E_4', 'PX_4', 'PY_4', 'PZ_4', 'E_5', 'PX_5', 'PY_5', 'PZ_5', 'E_6', 'PX_6', 'PY_6', 'PZ_6', 'E_7', 'PX_7', 'PY_7', 'PZ_7', 'E_8', 'PX_8', 'PY_8', 'PZ_8', 'E_9', 'PX_9', 'PY_9', 'PZ_9', 'E_10', 'PX_10', 'PY_10', 'PZ_10', 'E_11', 'PX_11', 'PY_11', 'PZ_11', 'E_12', 'PX_12', 'PY_12', 'PZ_12', 'E_13', 'PX_13', 'PY_13', 'PZ_13', 'E_14', 'PX_14', 'PY_14', 'PZ_14', 'E_15', 'PX_15', 'PY_15', 'PZ_15', 'E_16', 'PX_16', 'PY_16', 'PZ_16', 'E_17', 'PX_17', 'PY_17', 'PZ_17', 'E_18', 'PX_18', 'PY_18', 'PZ_18', 'E_19', 'PX_19', 'PY_19', 'PZ_19', 'E_20', 'PX_20', 'PY_20', 'PZ_20', 'E_21', 'PX_21', 'PY_21', 'PZ_21', 'E_22', 'PX_22', 'PY_22', 'PZ_22', 'E_23', 'PX_23', 'PY_23', 'PZ_23', 'E_24', 'PX_24', 'PY_24', 'PZ_24', 'E_25', 'PX_25', 'PY_25', 'PZ_25', 'E_26', 'PX_26', 'PY_26', 'PZ_26', 'E_27', 'PX_27', 'PY_27', 'PZ_27', 'E_28', 'PX_28', 'PY_28', 'PZ_28',

In [16]:
import pandas as pd
import numpy as np

df = pd.read_hdf('/content/data/train.h5', key='table', stop=5)

# Reshape para (5, 200, 4) em ordem (E, PX, PY, PZ)
feat_cols = [f'{q}_{i}' for i in range(200) for q in ['E', 'PX', 'PY', 'PZ']]
X = df[feat_cols].values.reshape(5, 200, 4)
y = df['is_signal_new'].values

print("Feature shape:", X.shape)       # esperado: (5, 200, 4)
print("Labels:", y)                     # esperado: 0s e 1s
print("Signal fraction:", y.mean())

# Real particles per jet
n_real = (X[:,:,0] > 0).sum(axis=1)   # E > 0 indica particula real
print("Real particles per jet:", n_real)
print("Min/max:", n_real.min(), n_real.max())

# Padding check
padding_mask = X[:,:,0] == 0
print("All padding are zeros:", (X[padding_mask] == 0).all())

# ttv values
print("ttv unique:", df['ttv'].unique())

Feature shape: (5, 200, 4)
Labels: [0 0 0 0 0]
Signal fraction: 0.0
Real particles per jet: [23 43 41 25 73]
Min/max: 23 73
All padding are zeros: True
ttv unique: [0]


In [17]:
df_check = pd.read_hdf('/content/data/train.h5', key='table', stop=100000)
print("Signal fraction (10K sample):", df_check['is_signal_new'].mean())

Signal fraction (10K sample): 0.5004


In [18]:
import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import knn_graph
from sklearn.neighbors import NearestNeighbors

def build_graph(row: np.ndarray, k: int = 7) -> Data:
    """
    row: (200, 4) array with columns (E, PX, PY, PZ)
    returns: PyG Data object with node features and edge_index
    """
    # 1. Filter padding
    mask = row[:, 0] > 0          # E > 0
    particles = row[mask]          # (N_real, 4)

    E  = particles[:, 0]
    PX = particles[:, 1]
    PY = particles[:, 2]
    PZ = particles[:, 3]

    # 2. Convert to (pT, eta, phi, E)
    pT  = np.sqrt(PX**2 + PY**2)
    p   = np.sqrt(PX**2 + PY**2 + PZ**2)
    eta = np.arctanh(np.clip(PZ / (p + 1e-8), -1 + 1e-7, 1 - 1e-7))
    phi = np.arctan2(PY, PX)

    # 3. Relative normalization
    pT_sum = pT.sum() + 1e-8
    E_sum  = E.sum()  + 1e-8

    eta_jet = (pT * eta).sum() / pT_sum   # pT-weighted centroid
    phi_jet = (pT * phi).sum() / pT_sum

    pT_rel  = pT / pT_sum
    delta_eta = eta - eta_jet
    delta_phi = phi - phi_jet
    # wrap delta_phi to [-pi, pi]
    delta_phi = (delta_phi + np.pi) % (2 * np.pi) - np.pi
    E_rel   = E / E_sum

    # 4. Node features: (pT_rel, delta_eta, delta_phi, E_rel)
    x = np.stack([pT_rel, delta_eta, delta_phi, E_rel], axis=1)  # (N_real, 4)

   # 5. Build k-NN graph in (eta, phi) space usando sklearn
    pos_np = np.stack([delta_eta, delta_phi], axis=1)  # (N_real, 2)
    k_actual = min(k, len(pos_np) - 1)

    nbrs = NearestNeighbors(n_neighbors=k_actual + 1, algorithm='auto').fit(pos_np)
    _, indices = nbrs.kneighbors(pos_np)
    # indices[:, 0] e o proprio no, descarta
    src = np.repeat(np.arange(len(pos_np)), k_actual)
    dst = indices[:, 1:].flatten()
    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)

    x_t = torch.tensor(x, dtype=torch.float)
    return Data(x=x_t, edge_index=edge_index)

In [19]:
feat_cols = [f'{q}_{i}' for i in range(200) for q in ['E', 'PX', 'PY', 'PZ']]
df_sample = pd.read_hdf('/content/data/train.h5', key='table', stop=5)
X = df_sample[feat_cols].values.reshape(5, 200, 4)

g = build_graph(X[0])
print("Nodes:", g.num_nodes)
print("Edges:", g.num_edges)
print("Node features shape:", g.x.shape)    # esperado: (N_real, 4)
print("Edge index shape:", g.edge_index.shape)  # esperado: (2, N_real*k)

Nodes: 23
Edges: 161
Node features shape: torch.Size([23, 4])
Edge index shape: torch.Size([2, 161])


In [21]:
# One-time conversion — run once, then never again
import numpy as np
import pandas as pd

FEAT_COLS = [f'{q}_{i}' for i in range(200) for q in ['E', 'PX', 'PY', 'PZ']]

def convert_to_numpy(hdf_path: str, out_path: str, chunk_size: int = 50_000):
    store = pd.HDFStore(hdf_path, mode='r')
    total = store.get_storer('table').nrows
    store.close()

    X_all, y_all = [], []

    for start in range(0, total, chunk_size):
        df = pd.read_hdf(hdf_path, key='table', start=start, stop=start + chunk_size)
        X_all.append(df[FEAT_COLS].values.reshape(-1, 200, 4).astype(np.float32))
        y_all.append(df['is_signal_new'].values.astype(np.int64))
        print(f"  converted {min(start + chunk_size, total)}/{total}")
        del df

    np.save(out_path + '_X.npy', np.concatenate(X_all))
    np.save(out_path + '_y.npy', np.concatenate(y_all))
    print("Done.")

convert_to_numpy('/content/data/train.h5', '/content/data/train')
convert_to_numpy('/content/data/val.h5',   '/content/data/val')
convert_to_numpy('/content/data/test.h5',  '/content/data/test')

  converted 50000/1211000
  converted 100000/1211000
  converted 150000/1211000
  converted 200000/1211000
  converted 250000/1211000
  converted 300000/1211000
  converted 350000/1211000
  converted 400000/1211000
  converted 450000/1211000
  converted 500000/1211000
  converted 550000/1211000
  converted 600000/1211000
  converted 650000/1211000
  converted 700000/1211000
  converted 750000/1211000
  converted 800000/1211000
  converted 850000/1211000
  converted 900000/1211000
  converted 950000/1211000
  converted 1000000/1211000
  converted 1050000/1211000
  converted 1100000/1211000
  converted 1150000/1211000
  converted 1200000/1211000
  converted 1211000/1211000
Done.
  converted 50000/403000
  converted 100000/403000
  converted 150000/403000
  converted 200000/403000
  converted 250000/403000
  converted 300000/403000
  converted 350000/403000
  converted 400000/403000
  converted 403000/403000
Done.
  converted 50000/404000
  converted 100000/404000
  converted 150000/40400

In [26]:
import numpy as np
import torch
from torch_geometric.data import Dataset # Added import for Dataset

class JetDataset(Dataset):
    def __init__(self, prefix: str, n_samples: int = None):
        super().__init__()
        self.X = np.load(prefix + '_X.npy', mmap_mode='r')
        self.y = np.load(prefix + '_y.npy')
        self.n_samples = n_samples or len(self.y)

    def len(self):
        return self.n_samples

    def get(self, idx):
        graph = build_graph(self.X[idx])
        graph.y = torch.tensor([self.y[idx]], dtype=torch.long)
        return graph

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool

class JetGIN(nn.Module):
    def __init__(self):
        super().__init__()

        def mlp(in_ch, out_ch):
            return nn.Sequential(
                nn.Linear(in_ch, out_ch),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
                nn.Linear(out_ch, out_ch),
            )

        self.conv1 = GINConv(mlp(4,  64))
        self.conv2 = GINConv(mlp(64, 64))
        self.conv3 = GINConv(mlp(64, 64))

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2),
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

In [28]:
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset # Ensure Dataset is imported if JetDataset is defined elsewhere

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = JetGIN().to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

# Forward pass com um batch
# Create a small JetDataset instance for testing purposes
dataset_for_test = JetDataset('/content/data/train', n_samples=4) # Use a small sample size for quick testing
batch = next(iter(DataLoader(dataset_for_test, batch_size=4)))
batch = batch.to(device)
out = model(batch.x, batch.edge_index, batch.batch)
print("Output shape:", out.shape)   # esperado: (4, 2)

Device: cuda
JetGIN(
  (conv1): GINConv(nn=Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (conv2): GINConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (conv3): GINConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import (
    GCNConv, GATConv, EdgeConv, TransformerConv,
    GINConv, global_mean_pool
)

# ── shared classifier head ──────────────────────────────────────────
def classifier_head():
    return nn.Sequential(
        nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2)
    )

# ── GCN ────────────────────────────────────────────────────────────
class JetGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 64)
        #self.conv3 = GCNConv(64, 64)
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        #x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# ── GAT ────────────────────────────────────────────────────────────
class JetGAT(nn.Module):
    def __init__(self, heads=4):
        super().__init__()
        self.conv1 = GATConv(4,  16, heads=heads, concat=True)   # -> 64
        self.conv2 = GATConv(64, 16, heads=heads, concat=True)   # -> 64
        #self.conv3 = GATConv(64, 64, heads=1,     concat=False)  # -> 64
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        #x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# ── EdgeConv (ParticleNet-style) ────────────────────────────────────
# MLP recebe (x_i || x_j - x_i), entao input = 2 * in_ch
class JetEdgeConv(nn.Module):
    def __init__(self):
        super().__init__()
        def edge_mlp(in_ch, out_ch):
            return nn.Sequential(
                nn.Linear(in_ch * 2, out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(),
                nn.Linear(out_ch,    out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(),
            )
        self.conv1 = EdgeConv(edge_mlp(4,  64))
        self.conv2 = EdgeConv(edge_mlp(64, 64))
        #self.conv3 = EdgeConv(edge_mlp(64, 64))
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.conv2(x, edge_index)
        #x = self.conv3(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# ── Particle Transformer (simplificado) ────────────────────────────
class JetTransformer(nn.Module):
    def __init__(self, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(4,  16, heads=heads, concat=True)   # -> 64
        self.conv2 = TransformerConv(64, 16, heads=heads, concat=True)   # -> 64
        #self.conv3 = TransformerConv(64, 64, heads=1,     concat=False)  # -> 64
        self.norm1 = nn.LayerNorm(64)
        self.norm2 = nn.LayerNorm(64)
        #self.norm3 = nn.LayerNorm(64)
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = self.norm1(F.relu(self.conv1(x, edge_index)))
        x = self.norm2(F.relu(self.conv2(x, edge_index)))
        #x = self.norm3(F.relu(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

In [52]:
models = {
    'GCN':         JetGCN(),
    'GIN':         JetGIN(),
    'GAT':         JetGAT(),
    'EdgeConv':    JetEdgeConv(),
    'Transformer': JetTransformer(),
    'DyEdgeConv': JetDynamicEdgeConv(),
}

# Using the dataset_for_test that we defined previously
batch = next(iter(DataLoader(dataset_for_test, batch_size=4))).to(device)

print(f"{'Model':<15} {'Params':>10} {'Output shape'}")
print("-" * 40)
for name, m in models.items():
    m = m.to(device)
    out = m(batch.x, batch.edge_index, batch.batch)
    n = sum(p.numel() for p in m.parameters())
    print(f"{name:<15} {n:>10,}   {tuple(out.shape)}")

Model               Params Output shape
----------------------------------------
GCN                  6,626   (4, 2)
GIN                 23,650   (4, 2)
GAT                  6,882   (4, 2)
EdgeConv            19,810   (4, 2)
Transformer         20,322   (4, 2)
DyEdgeConv          19,810   (4, 2)


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


In [53]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

class Trainer:
    def __init__(self, model, optimizer, scheduler=None, device='cpu'):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.best_val_loss = float('inf')
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_acc':  [], 'val_acc':  [], 'lr': []
        }

    def fit(self, train_loader, val_loader, epochs, checkpoint_path=None):
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self._train_epoch(train_loader)
            val_loss,   val_acc   = self._val_epoch(val_loader)

            if self.scheduler:
                self.scheduler.step(val_loss)

            lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['lr'].append(lr)

            saved = ''
            if checkpoint_path and val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.save(checkpoint_path)
                saved = '  [saved]'

            print(f"Epoch {epoch:03d} | "
                  f"train loss={train_loss:.4f} acc={train_acc:.4f} | "
                  f"val loss={val_loss:.4f} acc={val_acc:.4f} | "
                  f"lr={lr:.2e}{saved}")

        return self.history

    def _train_epoch(self, loader):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for batch in loader:
            batch = batch.to(self.device)
            self.optimizer.zero_grad()
            out  = self.model(batch.x, batch.edge_index, batch.batch)
            loss = self.criterion(out, batch.y.squeeze())
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            correct    += (out.argmax(1) == batch.y.squeeze()).sum().item()
            total      += batch.num_graphs
        return total_loss / total, correct / total

    def _val_epoch(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(self.device)
                out  = self.model(batch.x, batch.edge_index, batch.batch)
                loss = self.criterion(out, batch.y.squeeze())
                total_loss += loss.item() * batch.num_graphs
                correct    += (out.argmax(1) == batch.y.squeeze()).sum().item()
                total      += batch.num_graphs
        return total_loss / total, correct / total

    def evaluate(self, loader):
        self.model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(self.device)
                out   = self.model(batch.x, batch.edge_index, batch.batch)
                probs = torch.softmax(out, dim=1)[:, 1]
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(batch.y.squeeze().cpu().numpy())

        probs  = np.array(all_probs)
        labels = np.array(all_labels)

        return {
            'auc_roc':         roc_auc_score(labels, probs),
            'accuracy':        accuracy_score(labels, (probs > 0.5).astype(int)),
            'bg_rejection_30': self._bg_rejection(labels, probs, signal_eff=0.30),
            'bg_rejection_50': self._bg_rejection(labels, probs, signal_eff=0.50),
        }

    def _bg_rejection(self, labels, probs, signal_eff):
        threshold = np.percentile(probs[labels == 1], (1 - signal_eff) * 100)
        bg_eff = (probs[labels == 0] >= threshold).mean()
        return 1.0 / bg_eff if bg_eff > 0 else float('inf')

    def save(self, path):
        torch.save({
            'model_state':     self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'best_val_loss':   self.best_val_loss,
            'history':         self.history,
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        self.optimizer.load_state_dict(ckpt['optimizer_state'])
        self.best_val_loss = ckpt['best_val_loss']
        self.history       = ckpt['history']

In [57]:
import time
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader

# Fixed sample size for timing test
N_TRAIN  = 5_000
N_VAL    = 1_000
BATCH    =
N_EPOCHS = 3

train_ds = JetDataset('/content/data/train', n_samples=N_TRAIN)
val_ds   = JetDataset('/content/data/val',   n_samples=N_VAL)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

results = {}

for name, model_cls in [
    ('GCN',         JetGCN),
    ('GIN',         JetGIN),
    ('GAT',         JetGAT),
    ('EdgeConv',    JetEdgeConv),
    ('Transformer', JetTransformer),
    ('DyEdgeConv',  JetDynamicEdgeConv),
]:
    model     = model_cls().to(device)
    optimizer = Adam(model.parameters(), lr=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    trainer   = Trainer(model, optimizer, scheduler, device)

    epoch_times = []
    for epoch in range(1, N_EPOCHS + 1):
        t0 = time.perf_counter()
        train_loss, train_acc = trainer._train_epoch(train_loader)
        val_loss,   val_acc   = trainer._val_epoch(val_loader)
        elapsed = time.perf_counter() - t0
        epoch_times.append(elapsed)

    results[name] = {
        'params':        sum(p.numel() for p in model.parameters()),
        'time_mean_s':   sum(epoch_times) / len(epoch_times),
        'train_acc':     train_acc,
        'val_acc':       val_acc,
        'val_loss':      val_loss,
    }
    print(f"{name:>12} | {results[name]['time_mean_s']:.1f}s/epoch | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

# Summary table
print("\n" + "="*70)
print(f"{'Model':<14} {'Params':>8} {'s/epoch':>9} {'Projected 10ep':>16} {'val_acc':>9}")
print("-"*70)
for name, r in results.items():
    factor    = 1_211_000 / N_TRAIN          # scale to full dataset
    proj_10ep = r['time_mean_s'] * factor * 10
    proj_str  = f"{proj_10ep/3600:.1f}h" if proj_10ep > 3600 else f"{proj_10ep/60:.0f}min"
    print(f"{name:<14} {r['params']:>8,} {r['time_mean_s']:>8.1f}s "
          f"{proj_str:>16} {r['val_acc']:>8.4f}")

Device: cuda

         GCN | 8.3s/epoch | val_loss=0.6744 val_acc=0.6270
         GIN | 8.3s/epoch | val_loss=0.3726 val_acc=0.8340
         GAT | 7.8s/epoch | val_loss=0.6742 val_acc=0.5950
    EdgeConv | 8.4s/epoch | val_loss=0.3256 val_acc=0.8830
 Transformer | 8.4s/epoch | val_loss=0.3891 val_acc=0.8470
  DyEdgeConv | 15.3s/epoch | val_loss=0.3291 val_acc=0.8730

Model            Params   s/epoch   Projected 10ep   val_acc
----------------------------------------------------------------------
GCN               6,626      8.3s             5.6h   0.6270
GIN              23,650      8.3s             5.6h   0.8340
GAT               6,882      7.8s             5.3h   0.5950
EdgeConv         19,810      8.4s             5.7h   0.8830
Transformer      20,322      8.4s             5.7h   0.8470
DyEdgeConv       19,810     15.3s            10.3h   0.8730
